# ECE 232E — Spring 2026 — Project 2 Helper

In this helper notebook, the provided code for Q1–Q22 is written in R and the code for Q23–Q25 is written in Python. You are free to complete the project in whichever language you prefer.

## Setup (R + Python)

In [ ]:
!pip install rpy2==3.5.1

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
if (!require("ggplot2")) install.packages("ggplot2")
library(ggplot2)
if (!require("igraph"))  install.packages("igraph")
library(igraph)
if (!require("pracma"))  install.packages("pracma")
library(pracma)


---
# Part 1 — Facebook network (Q1 – Q17)

## Load the Facebook network

In [ ]:
%%R
fb_combined <- read.table("facebook_combined.txt")
tmp_mat <- as.matrix(fb_combined) + 1
g <- graph.edgelist(tmp_mat, directed = FALSE)


### Q1 — number of nodes / edges / connectivity

In [ ]:
%%R
fprintf('The number of nodes: %d\n', vcount(g))
fprintf('The number of edges: %d\n', ecount(g))
fprintf('Network Connected: %s\n', is_connected(g))


### Q2 — diameter

In [ ]:
%%R
diameter(g)


### Q3 — degree distribution and average degree

In [ ]:
%%R
y = degree(g)
mean(y)
h1 = hist(y, breaks=seq(0.0, by=1 , length.out=max(y)+2),
          xlab="Nodes", main="Histogram of degree distribution")
plot(degree.distribution(g), main="Degree distribution of the network",
     xlab="Degree", ylab="Frequency")


### Q4 — log-log degree distribution

### Q5 — personalized network of user ID 1

In [ ]:
%%R
network_id1 = make_ego_graph(g, order=1, nodes = V(g)[1])[[1]]
plot(network_id1, vertex.size=5, vertex.label=NA,
     main="Personalized network of the user with ID 1")
fprintf('The number of nodes: %d\n', vcount(network_id1))
fprintf('The number of edges: %d\n', ecount(network_id1))


### Q6 — diameter of the personalized network

### Q7 — interpretation of the diameter bounds

### Q8 — core nodes (degree > 200) and their average degree

In [ ]:
%%R
dg <- degree(g)
nodes_list <- dg > 200
core_nodes <- which(nodes_list == TRUE)
no_of_core_nodes <- length(core_nodes)
avg_core_node_degree <- mean(dg[core_nodes])
print(avg_core_node_degree)
print(no_of_core_nodes)


### Q9 — community detection on core node's personalized network

In [ ]:
%%R
g_ego = make_ego_graph(g, order=1, nodes=c('1'))[[1]]


In [ ]:
%%R
imc <- infomap.community(g_ego)


In [ ]:
%%R
fgc <- fastgreedy.community(g_ego)


In [ ]:
%%R
ebc <- edge.betweenness.community(g_ego)


In [ ]:
%%R
core_network = induced.subgraph(g, c(1, neighbors(g, 1)))
fast_greedy_com = fastgreedy.community(core_network)
fprintf('Fast-Greedy: %f\n', modularity(fast_greedy_com))
plot(core_network, mark.groups=groups(fast_greedy_com),
     vertex.size=5, vertex.label=NA, vertex.color=fast_greedy_com$membership,
     main=sprintf("Community structure of Fast-Greedy (Core node %s)", 1))


### Q10 — community structure with the core node removed

### Q11 — embeddedness vs. degree expression

### Q12 — embeddedness and dispersion histograms

### Q13 — highlight max-dispersion node

### Q14 — highlight max-embeddedness and max(dispersion/embeddedness) nodes

### Q15 — interpret the two measures

### Q16 — personalized network of node 415, build Nr (degree = 24)

In [ ]:
%%R
#graph_ego_415 =
#degree_list = degree(graph_ego_415)
#Nr = which(degree_list == 24)
#print(length(Nr))


### Q17 — friend recommendation accuracy (Common Neighbors / Jaccard / Adamic-Adar)

In [ ]:
%%R
#neighbors_i = neighbors(graph_ego_415, node_i)
#not_neighbors_i = setdiff(V(graph_ego_415), neighbors_i)
#metric = c()
#for nn in not_neighbors_i:
  #neighbors_nn = neighbors(graph_ego_415, nn)
  #neighbors_intersection = intersect(neighbors_i, neighbors_nn) #(get union similarly)  #similarity metric
  #metric = c(metric, length(neighbors_intersection))
#sort_index = sort(metric,decreasing=TRUE, index.return=TRUE)
#P = not_neighbors_i[index[1:t]]


---
# Part 2 — Google+ network (Q18 – Q22)

## Setup + build personal networks (users with > 2 circles)

In [ ]:
%%R
file_path ="gplus/"
edge_files = list.files(path=file_path,pattern="edges")
circles_files = list.files(path=file_path,pattern="circles")
fts_files = list.files(path=file_path,pattern="feat")
initial_graph = list()
final_graph = list()
graph_circles = list()
ego_nodes = list()


In [ ]:
%%R
cnt = 0
node_names = c()
for(i in 1:length(edge_files)){
    # get node id
    node = strsplit(edge_files[i],".edges")[[1]]
    node_names <-c(node_names,node)
    #print(node)
    ego_nodes[i] = node
    fc = file(paste(file_path,node,".circles",sep=""),open="r")
    if(length(fc)>0){
        file_lines <- readLines(fc)
        if(length(file_lines)>0){
            circles =list()
            for(j in 1:length(file_lines)){
                circle_users = strsplit(file_lines[j],"\t")
                circles[[j]] <- circle_users[[1]][-1]
              }
            # find users who have more than 2 circles
            if(length(circles)>2){
                cnt = cnt + 1
                initial_graph[[i]] <- read.graph(paste(file_path,edge_files[i],sep=""),format="ncol",directed=TRUE)
                graph_circles[[i]] <- circles
                graph_nodes <- V(initial_graph[[i]])
                print(length(graph_nodes))
                print(node)
                # add the core node to his neighbor list and construct the graph
                final_graph[[i]] <- add.vertices(initial_graph[[i]],1,name=node)
                core_index = which(V(final_graph[[i]])$name==node)
                core_node_edges = list()
                ### add edges connecting to this core node
                for(k in 1:length(graph_nodes)){
                    core_node_edges = c(core_node_edges, c(core_index, k))
                }
                final_graph[[i]] <- add.edges(final_graph[[i]],core_node_edges)
            }
        }
    }
    close(fc)
}


### Q18 — number of personal networks

In [ ]:
%%R
#q18
cat("there are ", length(edge_files),"nodes and there are ",cnt,"personal networks" )


### Q19 — in-degree and out-degree distributions (3 personal networks)

In [ ]:
%%R
#q19
interest_node = c('109327480479767108490', '115625564993990145546','101373961279443806744')
graph_inds = c()
for (i in 1: length(interest_node)){
    graph_ind <- which(node_names==interest_node[i])
    graph_inds <- c(graph_inds, graph_ind)
    print(graph_ind)
    tmp_graph = final_graph[[graph_ind]]
    hist(degree(tmp_graph, mode="in"),main = paste("in degree for ", interest_node[i]))
    hist(degree(tmp_graph, mode="out"),main = paste("out degree for ", interest_node[i]))
}


### Q20 — Walktrap community detection

In [ ]:
%%R
#q20
#https://igraph.org/r/doc/cluster_walktrap.html


### Q21 — meaning of homogeneity and completeness

### Q22 — compute h and c for the 3 personal networks

In [ ]:
%%R
# libraries are allowed. try to do without libraries?


---
# Part 3 — Cora dataset (Q23 – Q25)

## Q23 — Idea 1: Graph Convolutional Networks

In [ ]:
#feel free to use any other library of your choice
!pip install spektral
!pip install Keras
!pip install tensorflow


In [ ]:
import numpy as np
import os
import networkx as nx
from keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
from sklearn.metrics import classification_report

from spektral.layers import GCNConv

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dropout, Dense
from tensorflow.keras import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping
import tensorflow as tf
from tensorflow.keras.regularizers import l2

from collections import Counter
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt


In [ ]:
!unzip "cora (extract.me).zip"


In [ ]:
all_data = []
all_edges = []

for root,dirs,files in os.walk('./cora'):
    for file in files:
        if '.content' in file:
            with open(os.path.join(root,file),'r') as f:
                all_data.extend(f.read().splitlines())
        elif 'cites' in file:
            with open(os.path.join(root,file),'r') as f:
                all_edges.extend(f.read().splitlines())



all_data = shuffle(all_data,random_state=42)


In [ ]:
labels = []
nodes = []
X = []

for i,data in enumerate(all_data):
    elements = data.split('\t')
    labels.append(elements[-1])
    X.append(elements[1:-1])
    nodes.append(elements[0])

X = np.array(X,dtype=int)
N = X.shape[0]
F = X.shape[1]
print('X shape: ', X.shape)


#parse the edge
edge_list=[]
for edge in all_edges:
    e = edge.split('\t')
    edge_list.append((e[0],e[1]))

print('\nNumber of nodes (N): ', N)
print('\nNumber of features (F) of each node: ', F)
print('\nCategories: ', set(labels))

num_classes = len(set(labels))
print('\nNumber of classes: ', num_classes)


In [ ]:
G = nx.Graph()
G.add_nodes_from(nodes)
G.add_edges_from(edge_list)


A = nx.adjacency_matrix(G)
print('Graph info: ', nx.info(G))

#use gcc if you want.


In [ ]:
#get 20 train instances per class
#remaining instances are in test set test


In [ ]:
# Parameters
#channels, dropout_prob, learning rate, epochs, etc




# Model definition

#X_in = Input(shape=(F, ))
#fltr_in = Input((N, ), sparse=True)

#DEFINE YOUR MODEL HERE

# Build model
#model = Model(inputs ..., outputs=...)
#optimizer = # your optimizer
#model.compile(...)
#model.summary()


In [ ]:
# Train model
#validation_data =
#model.fit(...)

# Evaluate model

#prepare input for prediction
#y_pred = model.predict



#report = classification_report(np.argmax(y_te,axis=1), np.argmax(y_pred,axis=1), target_names=classes)
#print('GCN Classification Report: \n {}'.format(report))


In [ ]:
# layer_outputs = [layer.output for layer in model.layers]
# activation_model = Model(inputs=model.input, outputs=layer_outputs)
# activations = activation_model.predict([X,A],batch_size=N)


# x_tsne = TSNE(n_components=2).fit_transform(activations[3])

# def plot_tSNE(labels_encoded,x_tsne):
#     color_map = np.argmax(labels_encoded, axis=1)
#     plt.figure(figsize=(10,10))
#     for cl in range(num_classes):
#         indices = np.where(color_map==cl)
#         indices = indices[0]
#         plt.scatter(x_tsne[indices,0], x_tsne[indices, 1], label=cl)
#     plt.legend()
#     plt.show()

# plot_tSNE(labels_encoded,x_tsne)


## Q24 — Idea 2: Node2Vec

In [ ]:
# install StellarGraph if running on Google Colab
import sys
if 'google.colab' in sys.modules:
  %pip install -q stellargraph[demos]==1.2.1 --ignore-requires-python


In [ ]:
import matplotlib.pyplot as plt

from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import accuracy_score

import os
import networkx as nx
import numpy as np
import pandas as pd

from stellargraph.data import BiasedRandomWalk
from stellargraph import StellarGraph #feel free to use any other library of your choice
from stellargraph import datasets
from IPython.display import display, HTML

%matplotlib inline


In [ ]:
dataset = datasets.Cora()
display(HTML(dataset.description))
G, node_subjects = dataset.load(largest_connected_component_only=True)
print(G.info())


### Node2Vec

In [ ]:
# walks = get random walks using BiasedRandomWalk
#experiment with the hyperparamenters


In [ ]:
# from gensim.models import Word2Vec

# str_walks = [[str(n) for n in walk] for walk in walks]
# model = Word2Vec(str_walks, ) #check which parameter are important


In [ ]:
# The embedding vectors can be retrieved from model.wv using the node ID as key.
#model.wv["19231"].shape


### Visualize

In [ ]:
# Retrieve node embeddings and corresponding subjects
node_ids = model.wv.index_to_key  # list of node IDs
node_embeddings = (
    model.wv.vectors
)  # numpy.ndarray of size number of nodes times embeddings dimensionality
node_targets = node_subjects[[int(node_id) for node_id in node_ids]]


In [ ]:
# Apply t-SNE transformation on node embeddings
tsne = TSNE(n_components=2)
node_embeddings_2d = tsne.fit_transform(node_embeddings)


In [ ]:
# draw the points
alpha = 0.7
label_map = {l: i for i, l in enumerate(np.unique(node_targets))}
node_colours = [label_map[target] for target in node_targets]

plt.figure(figsize=(10, 8))
plt.scatter(
    node_embeddings_2d[:, 0],
    node_embeddings_2d[:, 1],
    c=node_colours,
    cmap="jet",
    alpha=alpha,
)


### CORA classification

In [ ]:
#X = None
#y = None
#split the data. use 140 examples in train set

#clf = classifier, use kfold cross validation
#clf.fit(X_train, y_train)
#y_pred = clf.predict(X_test)

#get accuracy, classification report


## Q25 — Idea 3: Personalized PageRank

In [ ]:
import numpy as np
import os
import networkx as nx
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from numpy import dot
from numpy.linalg import norm


from collections import Counter
import matplotlib.pyplot as plt
!unzip "cora (extract.me).zip"


In [ ]:
all_data = []
all_edges = []

for root,dirs,files in os.walk('./cora'):
    for file in files:
        if '.content' in file:
            with open(os.path.join(root,file),'r') as f:
                all_data.extend(f.read().splitlines())
        elif 'cites' in file:
            with open(os.path.join(root,file),'r') as f:
                all_edges.extend(f.read().splitlines())


#random_state = 42
#all_data = shuffle(all_data,random_state=random_state)


In [ ]:
categories =  ['Reinforcement_Learning', 'Theory', 'Case_Based', 'Genetic_Algorithms', 'Probabilistic_Methods', 'Neural_Networks', 'Rule_Learning']
sorted(categories)
label_encoder = {}
i = 0
for cat in sorted(categories):
  label_encoder[cat] = i
  i +=1
label_encoder


In [ ]:
#parse the data
labels = []
nodes = []
X = []
element_to_ind  = {}

for i,data in enumerate(all_data):
    elements = data.split('\t')
    labels.append(label_encoder[elements[-1]])
    X.append(elements[1:-1])
    nodes.append(elements[0])
    element_to_ind[elements[0]]= i
X = np.array(X,dtype=int)
N = X.shape[0] #the number of nodes
F = X.shape[1] #the size of node features
print('X shape: ', X.shape)


#parse the edge
edge_list=[]
for edge in all_edges:
    e = edge.split('\t')
    edge_list.append((e[0],e[1]))

print('\nNumber of nodes (N): ', N)
print('\nNumber of features (F) of each node: ', F)
print('\nCategories: ', set(labels))

num_classes = len(set(labels))
print('\nNumber of classes: ', num_classes)


In [ ]:
G = nx.Graph()
G.add_nodes_from(nodes)
G.add_edges_from(edge_list)
G = nx.relabel_nodes(G, element_to_ind)
print('Graph info: ', nx.info(G))


In [ ]:
nodes = list(G.nodes)
print(len(nodes))
list(G.neighbors(0))


In [ ]:
df = pd.DataFrame(list(zip(nodes, labels,X)),columns =['node', 'label','features'])
print(len(df))
df.head()


In [ ]:
Gcc = sorted(nx.connected_components(G), key=len, reverse=True)
G = G.subgraph(Gcc[0])
gcc_nodes = list(G.nodes)


In [ ]:
df = df.loc[df['node'].isin(gcc_nodes)]
df['node'] = list(range(len(df))) #rename nodes
df.head()


In [ ]:
train = df.groupby('label', group_keys=False).apply(lambda x: x.sample(20))
G = nx.relabel_nodes(G, df['node'])


In [ ]:
def create_transition_matrix(g):
    vs = list(g.nodes)
    n = len(vs)
    adj = nx.adjacency_matrix(g)
    transition_matrix = adj/adj.sum(axis=1)

    return transition_matrix


In [ ]:
def random_walk(g, num_steps, start_node, transition_matrix = None):
  if transition_matrix is None:
    transition_matrix = create_transition_matrix(g)
  #perform a random walk
  #return v


In [ ]:
seeds_dict = {predicted:list(train[train['label'] == predicted]['node']) for predicted in range(7)}

def random_walk_with_teleportation(g, num_steps, start_node,tp,predicted, transition_matrix = None):
  #modify random walk code to add teleportation.
  #you can only teleport to a node belonging to the same class as the start node


  #return v


In [ ]:
#pagerank. NO teleportation, NO tfidf.
transition_matrix = create_transition_matrix(G)

num_samples = 1000
num_walk_steps = 100

visiting_freq_label = []
for i in range(transition_matrix.shape[0]):
  visiting_freq_label.append([0,0,0,0,0,0,0])

visiting_freq = [0 for i in range(transition_matrix.shape[0])]


for train_node,predicted in zip(train['node'],train['label']):
  #print (train_node,predicted)
  for i in range(num_samples):
      start_point = train_node
      end_node = random_walk(G, num_walk_steps, start_point, transition_matrix)
      visiting_freq_label[end_node][predicted] += 1
      visiting_freq[end_node] +=1


In [ ]:
count = 0 #these many nodes remain unvisited.
for vf in visiting_freq:
  if vf ==0:
    count+=1
print('unvisited = ', count)
visiting_freq_label = np.asarray(visiting_freq_label)
preds = np.argmax(visiting_freq_label,axis = 1)
print(classification_report(df['label'], preds))
accuracy_score(df['label'], preds)


In [ ]:
#pagerank. WITH telportation, without tfidf

#repeat above expeiment but this time use the teleportation random walk

#get metrics


In [ ]:
vs = list(G.nodes)
n = len(vs)
adj = nx.adjacency_matrix(G)
transition = np.zeros((len(G.nodes), len(G.nodes)))

#for n1 in nodes:
  #for n2 in nodes:
    # if there is an edge between n1 and n2:
      # cos_sim = compute cosine similarity between features of n1 and n2
      # transition[n1,n2] = np.exp(cos_sim) #neumerator of softmax. #why do we need softmax?
#divide the values in transition by denominator of softmax. how will you do this?


In [ ]:
#pagerank. Without teleportation. WITH TFIDF
transition_matrix = transition

#perfrom pagerank using our tf_idf based transition matrix
#use randon walk without teleporation
#get metrics


In [ ]:
#pagerank. WITH teleportation WITH TFIDF
transition_matrix = transition

#same as above, except use random walk with teleportation
#get metrics
